This script intends to figure out the density profile data structure and match halos with its mass and accretion rate.

In [2]:
import h5py
import os
import pandas as pd
import numpy as np
from func import *
import illustris_python as il

### TNG300 Hydro

In [31]:
sim ='TNG300/sim_205_1250_Hydro/'
snapnum = 78
halos_dir = f'../../result/DMhalo_density_profiles_phys/{sim}/snap_{snapnum}/final_densities/'
halos_list = os.listdir(halos_dir)
print(halos_list)

['bin-15-20.npy', 'bin-10-15.npy', 'bin-40-45.npy', 'bin-25-30.npy', 'bin-20-25.npy', 'bin-30-35.npy', 'bin-35-40.npy']


In [4]:
bin_start = min([string[4:6] for string in halos_list])
bin_end = max([string[7:9] for string in halos_list])
print(bin_start, bin_end)

10 45


In [5]:
# Load halo masses
halo_M, halo_R, bins, densities = [], [], [], []
for fname in sorted(halos_list):
    print(fname)
    data = np.load(os.path.join(halos_dir, fname), allow_pickle=True).item()
    print(data.keys())
    halo_M.append(data['halo_M_Mean200'])
    halo_R.append(data['halo_R_Mean200'])
    bins.append(data['radial_bins'])
    densities.append(data['densities'])
    print(data['h'], data['scale_factor'], data['z'], data['rho_c'])
halo_M = np.concatenate(halo_M)
halo_R = np.concatenate(halo_R)
bins = np.concatenate(bins)
densities = np.concatenate(densities)
print(halo_M.shape)
print(halo_R.shape)
print(bins.shape)
print(densities.shape)

bin-10-15.npy
dict_keys(['halo_R_Mean200', 'halo_M_Mean200', 'h', 'scale_factor', 'z', 'radial_bins', 'densities', 'rho_c'])
0.6774 0.7705836268786364 0.2977176845174465 174.00854784966512 solMass / kpc3
bin-15-20.npy
dict_keys(['halo_R_Mean200', 'halo_M_Mean200', 'h', 'scale_factor', 'z', 'radial_bins', 'densities', 'rho_c'])
0.6774 0.7705836268786364 0.2977176845174465 174.00854784966512 solMass / kpc3
bin-20-25.npy
dict_keys(['halo_R_Mean200', 'halo_M_Mean200', 'h', 'scale_factor', 'z', 'radial_bins', 'densities', 'rho_c'])
0.6774 0.7705836268786364 0.2977176845174465 174.00854784966512 solMass / kpc3
bin-25-30.npy
dict_keys(['halo_R_Mean200', 'halo_M_Mean200', 'h', 'scale_factor', 'z', 'radial_bins', 'densities', 'rho_c'])
0.6774 0.7705836268786364 0.2977176845174465 174.00854784966512 solMass / kpc3
bin-30-35.npy
dict_keys(['halo_R_Mean200', 'halo_M_Mean200', 'h', 'scale_factor', 'z', 'radial_bins', 'densities', 'rho_c'])
0.6774 0.7705836268786364 0.2977176845174465 174.0085478496

In [6]:
print(data['h'], data['z'])

0.6774 0.2977176845174465


In [7]:
# Load the halo masses snap index table
idx_table_dir = f'../../result/DMhalo_mass_table/sim_205_1250_Hydro/snap_idx_table.csv'
df_idx = pd.read_csv(idx_table_dir, index_col=0)

In [8]:
# Load the halo masses in the mass table at the same redshift
mass_table_dir = f'../../result/DMhalo_mass_table/{sim}/mass_table.csv'
df_mass = pd.read_csv(mass_table_dir, index_col=0)

In [15]:
# Just to check if the mass table matches the index table
global_idx = 608921

for iS, S in enumerate([8, 13, 17, 21, 25, 33, 40, 50, 67, 78, 99]):
    if pd.isnull(df_idx[f'global_idx_{global_idx}'].iloc[iS]):
        pass
    else:
        basePath =  '/n/holylfs05/LABS/hernquist_lab/IllustrisTNG/Runs/L%dn%dTNG'%(205,1250)+'/output/'
        mass = il.groupcat.loadHalos(basePath, S, fields='Group_M_Mean200')
       
        # check if mass equals to the mass in the mass table
        print(mass[int(df_idx[f'global_idx_{global_idx}'].iloc[iS])], df_mass[f'global_idx_{global_idx}'].iloc[iS])
del global_idx

nan
nan


In [20]:
# Load the accretion rate data
accret_table_dir = f'../../result/DMhalo_mass_table/{sim}/accretion_table.csv'
df_accret = pd.read_csv(accret_table_dir, index_col=0)

In [23]:
# Sort the accretion rate so it matches the column of the index table
df_accret_sort = df_accret[df_idx.columns]
# Check if two columns are equal
print(df_accret_sort.equals(df_accret))
del df_accret

True


In [25]:
# Sort the mass table in the same way
df_mass_sort = df_mass[df_idx.columns]
# Check if two columns are equal
print(df_mass_sort.equals(df_mass))
del df_mass

True


In [38]:
# Convert non nan entries in df_idx to int
df_idx = df_idx.fillna(-1)
df_idx = df_idx.astype(int) 
df_idx = df_idx.replace(-1, np.nan)

In [56]:
snap_mass = il.groupcat.loadHalos(basePath, snapnum, fields='Group_M_Mean200')
subset_idx = np.where((snap_mass >= 10**1) & (snap_mass < 10**1.5))[0]
data = np.load(os.path.join(halos_dir, 'bin-10-15.npy'), allow_pickle=True).item()
print(snap_mass[subset_idx].shape, data['halo_M_Mean200'].shape)

# Get the row data
snap_idx_table = df_idx.loc[f'snap_{snapnum}']
snap_accret_table = df_accret_sort.loc[f'snap_{snapnum}']

add_accret = []
from tqdm import tqdm
for idx in tqdm(subset_idx[:100]):
    find_idx = np.where(snap_idx_table == idx)[0]
    if len(find_idx) == 0:
        add_accret.append([-1])
    elif len(find_idx) == 1:
        add_accret.append(snap_accret_table[find_idx])
    else:
        add_accret.append(np.mean(snap_accret_table[find_idx]))
add_accret = np.concatenate(add_accret)
print(add_accret.shape)

(160330,) (160330,)


100%|██████████| 160330/160330 [00:21<00:00, 7572.19it/s]

(160675,)
